# SGB — Sistema de Gestión de Biblioteca
Versión simple con **funciones** (sin clases). Python 3.10+.
Los datos se guardan en archivos `.txt` con campos separados por `|`.

## Celda 0 — Configuración

In [ ]:
import os
from datetime import date, timedelta

# Carpeta local donde se guardan los archivos .txt
RUTA_BASE = 'datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

SEPARADOR = '|'
MULTA_POR_DIA = 1000  # COP por cada día de retraso

# tipo de usuario -> (días de préstamo, cupo de libros)
TIPOS_USUARIO = {
    'estudiante': (7, 3),
    'docente':    (15, 5),
    'externo':    (5, 2),
}

# Campos de cada archivo (en orden)
CAMPOS = {
    'usuarios':   ['documento', 'nombre', 'tipo', 'correo', 'telefono', 'estado'],
    'categorias': ['id', 'nombre'],
    'libros':     ['isbn', 'titulo', 'autor', 'id_categoria', 'anio', 'total', 'disponibles'],
    'prestamos':  ['id', 'documento', 'isbn', 'fecha_prestamo', 'fecha_limite', 'fecha_devolucion'],
    'multas':     ['id', 'id_prestamo', 'documento', 'dias', 'valor', 'pagada'],
    'reservas':   ['id', 'documento', 'isbn', 'fecha', 'estado'],
}

# Campos que se guardan como texto pero se usan como números
CAMPOS_ENTEROS = {'id', 'id_categoria', 'anio', 'total', 'disponibles',
                  'id_prestamo', 'dias', 'valor'}

## Sección 1 — Archivos (leer y guardar)
Cada registro es un diccionario. Las fechas se guardan como texto `AAAA-MM-DD`.

In [ ]:
def ruta(nombre):
    """Ruta del archivo .txt de una tabla."""
    return os.path.join(RUTA_BASE, nombre + '.txt')


def cargar(nombre):
    """Lee un archivo y retorna una lista de diccionarios."""
    if not os.path.exists(ruta(nombre)):
        return []
    registros = []
    with open(ruta(nombre), encoding='utf-8') as f:
        for linea in f:
            linea = linea.rstrip('\n')
            if not linea:
                continue
            valores = linea.split(SEPARADOR)
            reg = dict(zip(CAMPOS[nombre], valores))
            for campo in reg:
                if campo in CAMPOS_ENTEROS:
                    reg[campo] = int(reg[campo])
            registros.append(reg)
    return registros


def guardar(nombre, registros):
    """Sobrescribe el archivo con la lista de diccionarios."""
    with open(ruta(nombre), 'w', encoding='utf-8') as f:
        for reg in registros:
            f.write(SEPARADOR.join(str(reg[c]) for c in CAMPOS[nombre]) + '\n')


def buscar(registros, campo, valor):
    """Primer registro cuyo campo sea igual al valor, o None."""
    for reg in registros:
        if reg[campo] == valor:
            return reg
    return None


def siguiente_id(registros):
    """Máximo id + 1."""
    return max((r['id'] for r in registros), default=0) + 1


def hoy_texto(fecha=None):
    """Fecha como texto ISO (hoy si no se indica)."""
    return (fecha or date.today()).isoformat()

## Sección 2 — Usuarios, categorías y libros

In [ ]:
def registrar_usuario(documento, nombre, tipo, correo, telefono):
    usuarios = cargar('usuarios')
    tipo = tipo.strip().lower()
    if tipo not in TIPOS_USUARIO:
        raise ValueError(f"Tipo inválido: {tipo} (use estudiante, docente o externo)")
    if buscar(usuarios, 'documento', documento):
        raise ValueError(f"Ya existe un usuario con documento {documento}")
    usuarios.append({'documento': documento, 'nombre': nombre, 'tipo': tipo,
                     'correo': correo, 'telefono': telefono, 'estado': 'activo'})
    guardar('usuarios', usuarios)


def cambiar_estado_usuario(documento, estado):
    if estado not in ('activo', 'suspendido'):
        raise ValueError("Estado inválido (use activo o suspendido)")
    usuarios = cargar('usuarios')
    u = buscar(usuarios, 'documento', documento)
    if not u:
        raise ValueError("Usuario no encontrado")
    u['estado'] = estado
    guardar('usuarios', usuarios)


def agregar_categoria(nombre):
    categorias = cargar('categorias')
    for c in categorias:
        if c['nombre'].lower() == nombre.strip().lower():
            raise ValueError("La categoría ya existe")
    nueva = {'id': siguiente_id(categorias), 'nombre': nombre.strip()}
    categorias.append(nueva)
    guardar('categorias', categorias)
    return nueva['id']


def agregar_libro(isbn, titulo, autor, id_categoria, anio, ejemplares):
    libros = cargar('libros')
    if buscar(libros, 'isbn', isbn):
        raise ValueError(f"Ya existe un libro con ISBN {isbn}")
    if not buscar(cargar('categorias'), 'id', int(id_categoria)):
        raise ValueError("La categoría no existe")
    if int(ejemplares) <= 0:
        raise ValueError("Debe haber al menos 1 ejemplar")
    libros.append({'isbn': isbn, 'titulo': titulo, 'autor': autor,
                   'id_categoria': int(id_categoria), 'anio': int(anio),
                   'total': int(ejemplares), 'disponibles': int(ejemplares)})
    guardar('libros', libros)


def buscar_libros(texto):
    """Libros cuyo título o autor contiene el texto."""
    t = texto.strip().lower()
    return [l for l in cargar('libros')
            if t in l['titulo'].lower() or t in l['autor'].lower()]


def texto_libro(l):
    return (f"{l['isbn']} | {l['titulo']} | {l['autor']} | cat {l['id_categoria']} | "
            f"{l['anio']} | disponibles {l['disponibles']}/{l['total']}")

## Sección 3 — Préstamos, devoluciones, multas y reservas
Reglas: usuario activo, sin multas pendientes y dentro de su cupo. Multa de COP 1.000 por día de retraso.
Las reservas se atienden por orden de llegada.

In [ ]:
def prestar_libro(documento, isbn, fecha=None):
    """Registra un préstamo y retorna su id."""
    usuarios, libros = cargar('usuarios'), cargar('libros')
    prestamos, multas, reservas = cargar('prestamos'), cargar('multas'), cargar('reservas')

    u = buscar(usuarios, 'documento', documento)
    l = buscar(libros, 'isbn', isbn)
    if not u:
        raise ValueError("Usuario no encontrado")
    if not l:
        raise ValueError("Libro no encontrado")
    if u['estado'] != 'activo':
        raise ValueError("El usuario está suspendido")
    if any(m['documento'] == documento and m['pagada'] == 'no' for m in multas):
        raise ValueError("El usuario tiene multas pendientes")

    dias, cupo = TIPOS_USUARIO[u['tipo']]
    activos = [p for p in prestamos if p['documento'] == documento and p['fecha_devolucion'] == '']
    if len(activos) >= cupo:
        raise ValueError(f"El usuario alcanzó su cupo de {cupo} libros")
    if l['disponibles'] <= 0:
        raise ValueError("No hay ejemplares disponibles (puede reservarlo)")

    # Respeta la cola de reservas: si hay reservas, solo presta al primero
    cola = [r for r in reservas if r['isbn'] == isbn and r['estado'] == 'activa']
    if cola:
        if len(cola) >= l['disponibles'] and cola[0]['documento'] != documento:
            raise ValueError("El libro está reservado por otro usuario")
        if cola[0]['documento'] == documento:
            cola[0]['estado'] = 'atendida'
            guardar('reservas', reservas)

    fecha = fecha or date.today()
    nuevo = {'id': siguiente_id(prestamos), 'documento': documento, 'isbn': isbn,
             'fecha_prestamo': hoy_texto(fecha),
             'fecha_limite': hoy_texto(fecha + timedelta(days=dias)),
             'fecha_devolucion': ''}
    prestamos.append(nuevo)
    l['disponibles'] -= 1
    guardar('prestamos', prestamos)
    guardar('libros', libros)
    return nuevo['id']


def devolver_libro(id_prestamo, fecha=None):
    """Registra la devolución. Retorna el valor de la multa (0 si no hay)."""
    prestamos, libros, multas = cargar('prestamos'), cargar('libros'), cargar('multas')
    p = buscar(prestamos, 'id', int(id_prestamo))
    if not p:
        raise ValueError("Préstamo no encontrado")
    if p['fecha_devolucion'] != '':
        raise ValueError("Ese préstamo ya fue devuelto")

    fecha = fecha or date.today()
    p['fecha_devolucion'] = hoy_texto(fecha)
    l = buscar(libros, 'isbn', p['isbn'])
    if l['disponibles'] < l['total']:
        l['disponibles'] += 1

    retraso = (fecha - date.fromisoformat(p['fecha_limite'])).days
    valor = 0
    if retraso > 0:
        valor = retraso * MULTA_POR_DIA
        multas.append({'id': siguiente_id(multas), 'id_prestamo': p['id'],
                       'documento': p['documento'], 'dias': retraso,
                       'valor': valor, 'pagada': 'no'})
        guardar('multas', multas)

    guardar('prestamos', prestamos)
    guardar('libros', libros)
    return valor


def pagar_multa(id_multa):
    multas = cargar('multas')
    m = buscar(multas, 'id', int(id_multa))
    if not m:
        raise ValueError("Multa no encontrada")
    if m['pagada'] == 'si':
        raise ValueError("La multa ya estaba pagada")
    m['pagada'] = 'si'
    guardar('multas', multas)


def reservar_libro(documento, isbn):
    """Crea una reserva (solo si no hay ejemplares). Retorna el id."""
    reservas = cargar('reservas')
    if not buscar(cargar('usuarios'), 'documento', documento):
        raise ValueError("Usuario no encontrado")
    l = buscar(cargar('libros'), 'isbn', isbn)
    if not l:
        raise ValueError("Libro no encontrado")
    if l['disponibles'] > 0:
        raise ValueError("El libro tiene ejemplares disponibles, puede prestarlo")
    for r in reservas:
        if r['documento'] == documento and r['isbn'] == isbn and r['estado'] == 'activa':
            raise ValueError("El usuario ya tiene una reserva activa de ese libro")
    nueva = {'id': siguiente_id(reservas), 'documento': documento, 'isbn': isbn,
             'fecha': hoy_texto(), 'estado': 'activa'}
    reservas.append(nueva)
    guardar('reservas', reservas)
    return nueva['id']


def cancelar_reserva(id_reserva):
    reservas = cargar('reservas')
    r = buscar(reservas, 'id', int(id_reserva))
    if not r or r['estado'] != 'activa':
        raise ValueError("Reserva activa no encontrada")
    r['estado'] = 'cancelada'
    guardar('reservas', reservas)

## Sección 4 — Historial y estadísticas

In [ ]:
def historial_usuario(documento):
    u = buscar(cargar('usuarios'), 'documento', documento)
    if not u:
        raise ValueError("Usuario no encontrado")
    print(f"\n--- Historial de {u['nombre']} ({u['tipo']}, {u['estado']}) ---")
    print("Préstamos:")
    for p in cargar('prestamos'):
        if p['documento'] == documento:
            dev = p['fecha_devolucion'] or 'pendiente'
            print(f"  #{p['id']} libro {p['isbn']} | {p['fecha_prestamo']} -> "
                  f"límite {p['fecha_limite']} | devuelto: {dev}")
    print("Multas:")
    for m in cargar('multas'):
        if m['documento'] == documento:
            print(f"  #{m['id']} préstamo {m['id_prestamo']} | {m['dias']} días | "
                  f"${m['valor']:,} | pagada: {m['pagada']}")
    print("Reservas:")
    for r in cargar('reservas'):
        if r['documento'] == documento:
            print(f"  #{r['id']} libro {r['isbn']} | {r['fecha']} | {r['estado']}")


def estadisticas():
    libros, usuarios = cargar('libros'), cargar('usuarios')
    prestamos, multas = cargar('prestamos'), cargar('multas')
    hoy = hoy_texto()

    activos = [p for p in prestamos if p['fecha_devolucion'] == '']
    vencidos = [p for p in activos if p['fecha_limite'] < hoy]
    devueltos = [p for p in prestamos if p['fecha_devolucion'] != '']

    print("\n--- Estadísticas ---")
    print(f"Libros: {len(libros)} títulos, {sum(l['total'] for l in libros)} ejemplares")
    print(f"Usuarios: {len(usuarios)}")
    print(f"Préstamos: {len(prestamos)} total | {len(activos)} activos | {len(vencidos)} vencidos")

    if devueltos:
        dur = [(date.fromisoformat(p['fecha_devolucion']) -
                date.fromisoformat(p['fecha_prestamo'])).days for p in devueltos]
        print(f"Duración promedio de préstamo: {sum(dur) / len(dur):.1f} días")

    # Libros más prestados
    conteo = {}
    for p in prestamos:
        conteo[p['isbn']] = conteo.get(p['isbn'], 0) + 1
    top = sorted(conteo.items(), key=lambda x: x[1], reverse=True)[:3]
    if top:
        print("Top libros más prestados:")
        for isbn, n in top:
            l = buscar(libros, 'isbn', isbn)
            print(f"  {l['titulo'] if l else isbn}: {n}")

    # Préstamos por tipo de usuario
    por_tipo = {t: 0 for t in TIPOS_USUARIO}
    for p in prestamos:
        u = buscar(usuarios, 'documento', p['documento'])
        if u:
            por_tipo[u['tipo']] += 1
    print("Préstamos por tipo de usuario:", por_tipo)

    pendiente = sum(m['valor'] for m in multas if m['pagada'] == 'no')
    recaudado = sum(m['valor'] for m in multas if m['pagada'] == 'si')
    print(f"Multas: ${pendiente:,} pendientes | ${recaudado:,} recaudadas")

## Prueba rápida
Usa una carpeta temporal, no toca los datos reales.

In [ ]:
import tempfile

_ruta_real = RUTA_BASE
RUTA_BASE = tempfile.mkdtemp()
try:
    hoy = date.today()
    registrar_usuario('1001', 'Ana Pérez', 'estudiante', 'ana@mail.com', '3001234567')
    registrar_usuario('1002', 'Luis Gómez', 'docente', 'luis@mail.com', '3007654321')
    cat = agregar_categoria('Ficción')
    agregar_libro('111', 'Cien años de soledad', 'García Márquez', cat, 1967, 2)
    agregar_libro('222', 'El coronel no tiene quien le escriba', 'García Márquez', cat, 1961, 1)
    assert len(buscar_libros('márquez')) == 2

    # Préstamo y devolución tardía (estudiante: 7 días) -> 3 días de multa
    p1 = prestar_libro('1001', '222', hoy - timedelta(days=10))
    assert devolver_libro(p1, hoy) == 3000
    # Con multa pendiente no puede pedir prestado
    try:
        prestar_libro('1001', '111'); raise AssertionError("debió fallar")
    except ValueError:
        pass
    pagar_multa(1)
    prestar_libro('1001', '111')

    # Reserva por orden de llegada
    prestar_libro('1002', '222')
    reservar_libro('1001', '222')
    devolver_libro(3)
    try:
        prestar_libro('1002', '222'); raise AssertionError("debió fallar")
    except ValueError:
        pass
    prestar_libro('1001', '222')  # la reserva queda atendida
    assert cargar('reservas')[0]['estado'] == 'atendida'

    historial_usuario('1001')
    estadisticas()
    print("\n✔ Todo OK")
finally:
    RUTA_BASE = _ruta_real

## Sección 5 — Menú principal

In [ ]:
def pedir(texto):
    return input(texto).strip()


def mostrar(lista, formato):
    if not lista:
        print("(sin registros)")
    for x in lista:
        print(formato(x))


def menu():
    opciones = """
====== SGB Biblioteca ======
 1. Registrar usuario
 2. Suspender / activar usuario
 3. Listar usuarios
 4. Agregar categoría
 5. Listar categorías
 6. Agregar libro
 7. Listar / buscar libros
 8. Prestar libro
 9. Devolver libro
10. Préstamos activos
11. Reservar libro
12. Cancelar reserva
13. Pagar multa
14. Historial de usuario
15. Estadísticas
 0. Salir"""
    while True:
        print(opciones)
        op = pedir("Opción: ")
        try:
            if op == '1':
                registrar_usuario(pedir("Documento: "), pedir("Nombre: "),
                                  pedir("Tipo (estudiante/docente/externo): "),
                                  pedir("Correo: "), pedir("Teléfono: "))
                print("Usuario registrado")
            elif op == '2':
                cambiar_estado_usuario(pedir("Documento: "), pedir("Estado (activo/suspendido): "))
                print("Estado actualizado")
            elif op == '3':
                mostrar(cargar('usuarios'), lambda u: f"{u['documento']} | {u['nombre']} | "
                        f"{u['tipo']} | {u['correo']} | {u['telefono']} | {u['estado']}")
            elif op == '4':
                print("Categoría creada con id", agregar_categoria(pedir("Nombre: ")))
            elif op == '5':
                mostrar(cargar('categorias'), lambda c: f"[{c['id']}] {c['nombre']}")
            elif op == '6':
                agregar_libro(pedir("ISBN: "), pedir("Título: "), pedir("Autor: "),
                              pedir("Id categoría: "), pedir("Año: "), pedir("Ejemplares: "))
                print("Libro agregado")
            elif op == '7':
                mostrar(buscar_libros(pedir("Texto a buscar (Enter = todos): ")), texto_libro)
            elif op == '8':
                n = prestar_libro(pedir("Documento: "), pedir("ISBN: "))
                print(f"Préstamo #{n} registrado")
            elif op == '9':
                valor = devolver_libro(pedir("Id del préstamo: "))
                print("Devolución registrada" + (f". Multa: ${valor:,} COP" if valor else ""))
            elif op == '10':
                mostrar([p for p in cargar('prestamos') if p['fecha_devolucion'] == ''],
                        lambda p: f"#{p['id']} | usuario {p['documento']} | libro {p['isbn']} | "
                                  f"límite {p['fecha_limite']}")
            elif op == '11':
                print("Reserva #", reservar_libro(pedir("Documento: "), pedir("ISBN: ")))
            elif op == '12':
                cancelar_reserva(pedir("Id de la reserva: "))
                print("Reserva cancelada")
            elif op == '13':
                pagar_multa(pedir("Id de la multa: "))
                print("Multa pagada")
            elif op == '14':
                historial_usuario(pedir("Documento: "))
            elif op == '15':
                estadisticas()
            elif op == '0':
                print("¡Hasta luego!")
                break
            else:
                print("Opción no válida")
        except ValueError as e:
            print("Error:", e)

## Ejecución

In [ ]:
menu()